# testing-llm-sense-of-direction

One board, one question, then a new game. The question is about the picture: which way the gold is, which way the agent is looking, and questions of that kind.

The menu is three base models, then every full save under `weights/full/`, newest first.

- Inkling 2 Mini (`thinkingmachines/Inkling-2-Mini`)
- Gemma 4 31B (`google/gemma-4-31B-it`)
- Gemma 4 12B (`google/gemma-4-12B-it`), the instruction-tuned checkpoint, not a play save

**Load** on a full save loads its Gemma trunk and its Localizer. The Localizer's s and v are added to the next token's embedding, the same way play does. S1 is not loaded, and a reply that ends in `[MOVE]` does not step the board. **Load** on one of the three base models uses that checkpoint alone: no s, no v, and nothing of that kind is stored.

**New game** deals with the same gold and opening choices as play. The system prompt is the play prompt loaded from NAMS. **Ask** sends the question once. The button stays off until the next new game. Neo4j has to be up.

Run this on the GPU box.


In [ ]:
import html
import io
import os
import sys
import traceback

os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import ipywidgets as widgets
from PIL import Image

from agent.notebook_ui import tame_shift_enter
from neural_net.direction_probe import DirectionProbe, direction_menu_options

state = {"probe": DirectionProbe(), "busy": False}


def _png(frame):
    image = Image.fromarray(frame, mode="RGB")
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return buf.getvalue()


def _say(text):
    log.append_stdout(text if text.endswith("\n") else text + "\n")


def _set_busy(flag):
    state["busy"] = flag
    probe = state["probe"]
    ask_btn.disabled = flag or probe.used or probe.game is None
    for widget in (model_menu, load_btn, gold, opening, new_btn, box):
        widget.disabled = flag


def on_load(_):
    if state["busy"]:
        return
    _set_busy(True)
    status.value = "Loading."
    try:
        info = state["probe"].load(model_menu.value)
        kind = "with the Localizer" if info["localizer"] else "no Localizer"
        status.value = "Loaded " + info["label"] + ", " + kind + "."
        _say(info["model_id"])
        _say(status.value)
    except Exception:
        status.value = "Load failed."
        _say(traceback.format_exc())
    finally:
        _set_busy(False)


def on_new(_):
    if state["busy"]:
        return
    _set_busy(True)
    status.value = "Dealing a board."
    try:
        dealt = state["probe"].new_game(gold.value, str(opening.value))
        board.value = _png(dealt["frame"])
        reply.value = "<pre></pre>"
        status.value = (
            "One question. %s gold(s), opening %s."
            % (dealt["n_gold"], dealt["opening"])
        )
        _say(status.value)
    except Exception:
        status.value = "New game failed."
        _say(traceback.format_exc())
    finally:
        _set_busy(False)


def on_ask(_):
    if state["busy"]:
        return
    probe = state["probe"]
    if probe.game is None:
        status.value = "Start a game first."
        return
    if probe.used:
        status.value = "Start a new game for another question."
        return
    question = box.value.strip()
    if not question:
        status.value = "Write a question."
        return
    _set_busy(True)
    status.value = "Asking."
    try:
        text = probe.ask(question)
        reply.value = "<pre>" + html.escape(text) + "</pre>"
        status.value = "Answered. Start a new game for another question."
        _say(status.value)
    except Exception:
        status.value = "Ask failed."
        _say(traceback.format_exc())
    finally:
        _set_busy(False)


model_menu = widgets.Dropdown(
    options=direction_menu_options(),
    value="inkling-2-mini",
    description="Model",
    layout=widgets.Layout(width="480px"),
)
load_btn = widgets.Button(description="Load", button_style="primary")
gold = widgets.Dropdown(
    options=[("random", None)] + [(str(n), n) for n in (0, 1, 2, 3)],
    value=None,
    description="Gold:",
    style={"description_width": "50px"},
    layout=widgets.Layout(width="160px"),
)
opening = widgets.Dropdown(
    options=[("require exit", "require"), ("any", "any"), ("sealed", "forbid")],
    value="any",
    description="Opening:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="220px"),
)
new_btn = widgets.Button(description="New game", button_style="info")
box = widgets.Textarea(
    value="Is the gold to the right or to the left of you?",
    layout=widgets.Layout(width="100%", height="80px"),
)
ask_btn = widgets.Button(description="Ask", button_style="success")
status = widgets.HTML("<span>Load a model, then start a game.</span>")
board = widgets.Image(format="png", width=420)
reply = widgets.HTML("<pre></pre>")
log = widgets.Output(layout=widgets.Layout(border="1px solid #ccc"))

load_btn.on_click(on_load)
new_btn.on_click(on_new)
ask_btn.on_click(on_ask)
ask_btn.disabled = True

box_ui = widgets.VBox([
    widgets.HBox([model_menu, load_btn]),
    widgets.HBox([gold, opening, new_btn]),
    widgets.HTML("<b>Board</b>"),
    board,
    box,
    ask_btn,
    status,
    widgets.HTML("<b>Reply</b>"),
    reply,
    log,
])
tame_shift_enter(box)
box_ui
